<a href="https://colab.research.google.com/github/pohlcriss/Aulas/blob/main/AtividadeAula09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!apt-get update -q
!apt-get install -y chromium-chromedriver -q
!pip install selenium -q

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:3 https://dl.google.com/linux/chrome-stable/deb stable InRelease [2,548 B]
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,411 B]
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 7,876 B in 1s (10.9 kB/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Readin

In [11]:
import sys
!{sys.executable} -m pip install -q requests

import requests
import unicodedata


def normalizar(txt):
    return unicodedata.normalize("NFD", txt).encode("ascii", "ignore").decode().lower().strip()


def buscar_municipio(nome):
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    achados = [m for m in resp.json() if normalizar(m["nome"]) == normalizar(nome)]

    if not achados:
        return None

    if len(achados) > 1:
        print("Mais de uma cidade encontrada:")
        for i, m in enumerate(achados):
            uf = m["microrregiao"]["mesorregiao"]["UF"]["sigla"]
            print(f"  {i} - {m['nome']}/{uf}")
        escolha = int(input("Escolha o número: "))
        return achados[escolha]

    return achados[0]


def buscar_previsao(codigo):
    url = f"https://apiprevmet3.inmet.gov.br/previsao/{codigo}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()[str(codigo)]


def extrair_temperaturas(periodos):
    if any(k in periodos for k in ("manha", "tarde", "noite")):
        blocos = [b for b in periodos.values() if isinstance(b, dict)]
    else:
        blocos = [periodos]

    mins = [int(b["temp_min"]) for b in blocos if b.get("temp_min")]
    maxs = [int(b["temp_max"]) for b in blocos if b.get("temp_max")]
    return (min(mins) if mins else None), (max(maxs) if maxs else None)


def main():
    cidade = input("Digite o nome da cidade: ")

    municipio = buscar_municipio(cidade)
    if not municipio:
        print("Cidade não encontrada.")
        return

    uf = municipio["microrregiao"]["mesorregiao"]["UF"]["sigla"]
    codigo = municipio["id"]

    try:
        previsao = buscar_previsao(codigo)
    except Exception as e:
        print(f"Erro ao consultar o INMET: {e}")
        return

    print("\n-----------------------------")
    print("      PREVISÃO DO TEMPO")
    print("-----------------------------")
    print(f"Cidade: {municipio['nome']}/{uf}\n")

    for data, periodos in previsao.items():
        try:
            tmin, tmax = extrair_temperaturas(periodos)
            print(f"{data}: mínima {tmin}°C | máxima {tmax}°C")
        except Exception:
            print(f"{data}: formato inesperado -> {periodos}")

    print("-----------------------------")


main()

Digite o nome da cidade: Horizontina

-----------------------------
      PREVISÃO DO TEMPO
-----------------------------
Cidade: Horizontina/RS

23/09/2026: mínima 7°C | máxima 20°C
24/09/2026: mínima 12°C | máxima 27°C
25/09/2026: mínima 16°C | máxima 30°C
26/09/2026: mínima 18°C | máxima 33°C
27/09/2026: mínima 21°C | máxima 34°C
-----------------------------
